# Bronze Layer Analysis (Parametrized)

This notebook is designed to be executed by Papermill.
Parameters are injected by Airflow at runtime.


In [ ]:
# --- Papermill parameters cell (tag: 'parameters') ---
# These values are defaults; Airflow overrides them at runtime.
execution_date = '2024-01-01'
bucket         = 'bronze'


In [ ]:
import os
import io
import boto3
import pandas as pd
from botocore.config import Config
from datetime import datetime

MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT', 'http://minio:9000')
MINIO_USER     = os.getenv('MINIO_ROOT_USER', 'minio_admin')
MINIO_PASS     = os.getenv('MINIO_ROOT_PASSWORD', 'minio_secure_pass_2024')

print(f'execution_date : {execution_date}')
print(f'bucket         : {bucket}')
print(f'run_at         : {datetime.now().isoformat()}')

In [ ]:
s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_USER,
    aws_secret_access_key=MINIO_PASS,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1',
)

year, month, day = execution_date[:4], execution_date[5:7], execution_date[8:]
key = f'sales/year={year}/month={month}/day={day}/orders.parquet'

try:
    resp = s3.get_object(Bucket=bucket, Key=key)
    df = pd.read_parquet(io.BytesIO(resp['Body'].read()))
    print(f'Loaded {len(df)} rows from s3://{bucket}/{key}')
except Exception as e:
    print(f'Data not yet available for {execution_date}: {e}')
    df = pd.DataFrame()

In [ ]:
if not df.empty:
    print('=== Schema ===')
    print(df.dtypes)
    print()
    print('=== Sample ===')
    display(df.head())
    print()
    print('=== Quality Summary ===')
    summary = {
        'total_rows'       : len(df),
        'null_ids'         : int(df['id'].isnull().sum()),
        'negative_prices'  : int((df['unit_price'] < 0).sum()),
        'total_revenue'    : round(float(df['total_price'].sum()), 2),
        'avg_order_value'  : round(float(df['total_price'].mean()), 2),
        'unique_customers' : int(df['customer_id'].nunique()),
        'unique_products'  : int(df['product_id'].nunique()),
    }
    for k, v in summary.items():
        print(f'  {k:25s}: {v}')
else:
    print('No data to analyze.')

In [ ]:
if not df.empty:
    top_products = (
        df.groupby('product_id')['total_price']
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
        .rename(columns={'total_price': 'revenue'})
    )
    print(f'Top 10 Products by Revenue ({execution_date})')
    display(top_products)